# Data Cleaning & Exploratory Analysis Notebook
## Model 1: Climate & Sustainability-Aware House Price Prediction

This notebook loads house price datasets for six major Indian cities (**Bengaluru, Chennai, Delhi, Hyderabad, Kolkata, Mumbai**), identifies and fixes data quality bugs (specifically the **"9" bug** in amenity columns), removes duplicate records, filters extreme outliers, performs feature engineering, and exports the clean dataset `cleaned_house_prices_6cities.csv`.

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

print('Libraries loaded successfully!')

Libraries loaded successfully!


### 1. Load and Merge City Datasets

In [2]:
files = {
    'Bengaluru': 'Bangalore.csv',
    'Chennai': 'Chennai.csv',
    'Hyderabad': 'Hyderabad.csv',
    'Kolkata': 'Kolkata.csv',
    'Mumbai': 'Mumbai.csv',
    'Delhi': 'Delhi.csv'
}

data = []
for city, file in files.items():
    temp = pd.read_csv(file)
    temp['City'] = city
    data.append(temp)

df_raw = pd.concat(data, ignore_index=True)
print(f'Initial Raw Shape: {df_raw.shape}')
print(df_raw['City'].value_counts())

Initial Raw Shape: (32963, 41)
City
Mumbai       7719
Kolkata      6507
Bengaluru    6207
Chennai      5014
Delhi        4998
Hyderabad    2518
Name: count, dtype: int64


### 2. Remove Duplicate Rows

In [3]:
print(f'Duplicate rows in raw data: {df_raw.duplicated().sum()}')
df_dedup = df_raw.drop_duplicates().reset_index(drop=True)
print(f'Shape after deduplication: {df_dedup.shape}')

Duplicate rows in raw data: 3828
Shape after deduplication: (29135, 41)


### 3. Diagnose and Fix the "9" Bug in Amenity Columns
In Kaggle Indian house price datasets, missing/unreported amenity features were encoded as `9`. Feeding `9` into ML models distorts predictions by treating missing values as 9 times present.

In [4]:
amenity_cols = [
    'MaintenanceStaff', 'Gymnasium', 'SwimmingPool', 'LandscapedGardens', 'JoggingTrack',
    'RainWaterHarvesting', 'IndoorGames', 'ShoppingMall', 'Intercom', 'SportsFacility',
    'ATM', 'ClubHouse', 'School', '24X7Security', 'PowerBackup', 'CarParking',
    'StaffQuarter', 'Cafeteria', 'MultipurposeRoom', 'Hospital', 'WashingMachine',
    'Gasconnection', 'AC', 'Wifi', "Children'splayarea", 'LiftAvailable', 'BED',
    'VaastuCompliant', 'Microwave', 'GolfCourse', 'TV', 'DiningTable', 'Sofa',
    'Wardrobe', 'Stadium'
]

# Count total 9s
total_9s = (df_dedup[amenity_cols] == 9).sum().sum()
print(f'Total "9" entries across amenity columns: {total_9s}')

# Fix 9 -> 0
df_clean = df_dedup.copy()
for col in amenity_cols:
    df_clean[col] = df_clean[col].replace(9, 0)

print('Successfully converted all "9" values to 0 (absent/unreported)!')

Total "9" entries across amenity columns: 731815
Successfully converted all "9" values to 0 (absent/unreported)!


### 4. Feature Engineering
- `amenity_count`: Total number of reported amenities.
- `price_per_sqft`: Property price divided by area.
- `area_per_bedroom`: Average sq.ft. per bedroom.

In [5]:
df_clean['amenity_count'] = df_clean[amenity_cols].sum(axis=1)
df_clean['price_per_sqft'] = df_clean['Price'] / df_clean['Area']
df_clean['area_per_bedroom'] = df_clean['Area'] / df_clean['No. of Bedrooms']

print(df_clean[['Price', 'Area', 'price_per_sqft', 'amenity_count']].describe())

              Price          Area  price_per_sqft  amenity_count
count  2.913500e+04  29135.000000    2.913500e+04   29135.000000
mean   1.195267e+07   1301.816475    1.005100e+04       2.364235
std    2.387647e+07    767.862339    1.915949e+04       5.249689
min    2.000000e+06    200.000000    2.410000e+02       0.000000
25%    4.179999e+06    870.000000    4.024300e+03       0.000000
50%    6.884999e+06   1137.000000    5.851756e+03       0.000000
75%    1.230000e+07   1504.000000    9.722222e+03       0.000000
max    8.546000e+08  16000.000000    1.076444e+06      35.000000


### 5. Filter Price and Area Outliers
Remove extreme data entry errors (e.g. 500 sq ft flat listed at ₹85 Crores).

In [6]:
mask = (
    (df_clean['price_per_sqft'] >= 500) &
    (df_clean['price_per_sqft'] <= 150000) &
    (df_clean['Area'] >= 200) &
    (df_clean['Area'] <= 10000) &
    (df_clean['Price'] >= 500000)
)

df_final = df_clean[mask].copy().reset_index(drop=True)
print(f'Final Cleaned Shape: {df_final.shape}')
print('\nCity-wise Cleaned Counts:')
print(df_final['City'].value_counts())

Final Cleaned Shape: (29039, 44)

City-wise Cleaned Counts:
City
Mumbai       6904
Kolkata      6257
Bengaluru    5513
Chennai      4305
Delhi        4062
Hyderabad    1998
Name: count, dtype: int64


### 6. Export Cleaned Dataset

In [7]:
output_csv = 'cleaned_house_prices_6cities.csv'
df_final.to_csv(output_csv, index=False)
print(f'Successfully saved {output_csv}!')

Successfully saved cleaned_house_prices_6cities.csv!
